# ⚡ Trader Cockpit – Live Execution Hub
**Workflow**: Engine Connectivity → Real-Time Exposure → Order Book Microstructure → Active Signals

---

In [ ]:
from qtrader.output.analyst import AnalystSession, RoleContext
import polars as pl
import numpy as np
import time
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import clear_output, display

session = AnalystSession(role=RoleContext.TRADER)
API_HOST = 'localhost'
API_PORT = 8000
PLOTLY_DARK = dict(template="plotly_dark", plot_bgcolor='#0f1117', paper_bgcolor='#0f1117')

## 1. System Health & Heartbeat
Monitoring engine latency and connectivity status.

In [ ]:
is_alive = session.ping_live_api(host=API_HOST, port=API_PORT)
if is_alive:
    status = session.connect_live_api(host=API_HOST, port=API_PORT)
    latency_ms = status.get('latency_ms', 12)
else:
    print("⚠️ Simulation Mode: API Disconnected")
    status = {'engine_status': 'CONNECTED', 'regime': 'BULL', 'uptime': '14h 22m', 'signals_active': 8}
    latency_ms = 42

def render_health(lat):
    color = "#34d399" if lat < 50 else "#fbbf24" if lat < 150 else "#ef4444"
    fig = go.Figure(go.Indicator(
        mode = "gauge+number",
        value = lat,
        title = {'text': "Engine Latency (ms)", 'font': {'size': 14}},
        gauge = {
            'axis': {'range': [None, 300], 'tickwidth': 1, 'tickcolor': "#94a3b8"},
            'bar': {'color': color},
            'steps' : [
                {'range': [0, 50], 'color': "#064e3b"},
                {'range': [50, 150], 'color': "#78350f"},
                {'range': [150, 300], 'color': "#7f1d1d"}
            ]
        }
    ))
    fig.update_layout(height=250, margin=dict(t=50, b=0, l=30, r=30), **PLOTLY_DARK)
    fig.show()

render_health(latency_ms)

## 2. Live Order Book (L2 Depth)
Visualising liquidity clusters and price walls.

In [ ]:
# Real L2 Order Book Data via AnalystSession
try:
    levels = 20
    book = session.get_live_orderbook("BTC-USD", limit=levels)
    bids = [b["price"] for b in book["bids"]]
    bid_sizes = [b["size"] for b in book["bids"]]
    asks = [a["price"] for a in book["asks"]]
    ask_sizes = [a["size"] for a in book["asks"]]
    mid = (bids[0] + asks[0]) / 2 if bids and asks else 0.0
except Exception as e:
    print(f"Failed to fetch live orderbook: {e}")
    # Fallback to zeros if API fails
    levels, mid = 20, 65000.0
    import numpy as np
    bids, asks = [mid]*levels, [mid]*levels
    bid_sizes, ask_sizes = [0]*levels, [0]*levels

## 3. Position & Exposure Dashboard
Monitoring net exposure vs global risk limits.

In [ ]:
exposure = 2.4 # current pos in BTC
max_limit = 5.0
utilization = (exposure / max_limit) * 100

fig_dash = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "indicator"}, {"type": "indicator"}]]
)

fig_dash.add_trace(go.Indicator(
    mode = "number+delta",
    value = exposure,
    title = {"text": "Net Exposure (BTC)"},
    delta = {'reference': 2.0, 'relative': False},
    domain = {'row': 0, 'column': 0}
))

fig_dash.add_trace(go.Indicator(
    mode = "gauge+number",
    value = utilization,
    title = {"text": "Risk Utilisation %"},
    gauge = {'axis': {'range': [0, 100]}, 'bar': {'color': "#8b5cf6"}},
    domain = {'row': 0, 'column': 1}
))

fig_dash.update_layout(height=250, **PLOTLY_DARK)
fig_dash.show()